# Bayesian Spatial Panel Models and Diagnostics

This notebook demonstrates the panel-model classes in `bayespecon`:
- Fixed effects / pooled panel models: `OLSPanelFE`, `SARPanelFE`, `SEMPanelFE`, `SDMPanelFE`, `SDEMPanelFE`
- Random effects panel models: `OLSPanelRE`, `SARPanelRE`, `SEMPanelRE`

It also demonstrates diagnostics inspired by the MATLAB reference code:
- `rdiagnose`-style influence diagnostics
- Breusch-Pagan test
- ARCH test
- Ljung-Box Q test
- Panel-specific diagnostics (`panel_residual_structure`, `pesaran_cd_test`)

## Panel Model Equations

Let observations be stacked by time, then unit (panel_g convention).

Fixed-effects / pooled classes:
- OLS panel: $y_{it} = x_{it}'\beta + a_i + \tau_t + \epsilon_{it}$
- SAR panel: $y_{it} = \rho (Wy)_{it} + x_{it}'\beta + a_i + \tau_t + \epsilon_{it}$
- SEM panel: $y_{it} = x_{it}'\beta + a_i + \tau_t + u_{it},\; u_{it}=\lambda (Wu)_{it}+\epsilon_{it}$
- SDM panel: $y_{it} = \rho (Wy)_{it} + x_{it}'\beta + (Wx)_{it}'\theta + a_i + \tau_t + \epsilon_{it}$
- SDEM panel: $y_{it} = x_{it}'\beta + (Wx)_{it}'\theta + a_i + \tau_t + u_{it},\; u_{it}=\lambda (Wu)_{it}+\epsilon_{it}$

Random-effects classes:
- OLS random effects: $y_{it} = x_{it}'\beta + \alpha_i + \epsilon_{it},\; \alpha_i \sim N(0, \sigma_\alpha^2)$
- SAR random effects: $y_{it} = \rho (Wy)_{it} + x_{it}'\beta + \alpha_i + \epsilon_{it},\; \alpha_i \sim N(0, \sigma_\alpha^2)$
- SEM random effects: $y_{it} = x_{it}'\beta + \alpha_i + u_{it},\; u_{it}=\lambda (Wu)_{it}+\epsilon_{it},\; \alpha_i \sim N(0, \sigma_\alpha^2)$

The `model` argument selects the pooled or fixed-effects specification for the FE classes:
- `0` pooled
- `1` unit FE
- `2` time FE
- `3` two-way FE

The RE classes are hierarchical models with unit-level random intercepts, so they internally use the pooled design and estimate `sigma_alpha` directly.

In [ ]:
import pathlib
import sys

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from libpysal.graph import Graph

repo_root = pathlib.Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from bayespecon import (
    OLSPanelFE,
    OLSPanelRE,
    SARPanelFE,
    SARPanelRE,
    SDEMPanelFE,
    SDMPanelFE,
    SEMPanelFE,
    SEMPanelRE,
    dgp,
)

az.style.use("arviz-white")


## Build a Balanced Panel Dataset

For pedagogy, we generate synthetic panel data from the `bayespecon.dgp` module on a regular polygon grid and then assemble a formula-ready DataFrame.


In [ ]:
# Generate base geometry via cross-sectional DGP, then simulate panel data via panel DGP.
rng = np.random.default_rng(2026)
xcols = ["poverty", "rev_rating", "num_spots", "crowded"]
ycol = "price_pp"

base_gdf = dgp.simulate_sar(n=8, seed=2026, create_gdf=True, geometry_type="polygon")
W = Graph.build_contiguity(base_gdf, rook=False).transform("r")
N = len(base_gdf)
T = 4
beta = np.array([1.5, -0.8, 0.6, 0.4, -0.5], dtype=float)

panel_sim = dgp.simulate_panel_sar_fe(
    N=N,
    T=T,
    rho=0.35,
    beta=beta,
    sigma=1.0,
    sigma_alpha=0.5,
    rng=rng,
    W=W,
)

panel = pd.DataFrame(
    {
        "unit": panel_sim["unit"],
        "time": panel_sim["time"],
        ycol: panel_sim["y"],
    }
)
for j, name in enumerate(xcols, start=1):
    panel[name] = panel_sim["X"][:, j]



## Shared Helpers

In [ ]:
def fit_panel_model(model_cls, formula, data, W, model=3, draws=300, tune=300, chains=2, seed=42):
    m = model_cls(
        formula=formula,
        data=data,
        W=W,
        unit_col="unit",
        time_col="time",
        model=model,
        logdet_method="eigenvalue",
    )

    idata = m.fit(
        draws=draws,
        tune=tune,
        chains=chains,
        target_accept=0.9,
        random_seed=seed,
        progressbar=False,
        idata_kwargs={"log_likelihood": True},
    )

    summary = m.summary(round_to=3)
    effects = m.spatial_effects()
    effects_df = pd.DataFrame({
        "feature": effects["feature_names"],
        "direct": effects["direct"],
        "indirect": effects["indirect"],
        "total": effects["total"],
    })

    return m, idata, summary, effects_df


def diagnostics_table(idata, var_names):
    cols = ["mean", "sd", "ess_bulk", "ess_tail", "r_hat"]
    return az.summary(idata, var_names=var_names, round_to=3)[cols]


def show_trace(idata, var_names, title):
    az.plot_trace(idata, var_names=var_names)
    plt.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()

## Fit: OLSPanelFE

In [ ]:
formula = "price_pp ~ poverty + rev_rating + num_spots + crowded"
ols_panel, idata_ols, summary_ols, effects_ols = fit_panel_model(
    OLSPanelFE, formula, panel, W, model=3
)
display(summary_ols)
display(effects_ols)
display(diagnostics_table(idata_ols, ["beta", "sigma"]))
show_trace(idata_ols, ["sigma"], "OLSPanelFE trace")

## Fit: SARPanelFE

In [ ]:
sar_panel, idata_sar, summary_sar, effects_sar = fit_panel_model(
    SARPanelFE, formula, panel, W, model=3
)
display(summary_sar)
display(effects_sar)
display(diagnostics_table(idata_sar, ["rho", "beta", "sigma"]))
show_trace(idata_sar, ["rho", "sigma"], "SARPanelFE trace")

## Fit: SEMPanelFE

In [ ]:
sem_panel, idata_sem, summary_sem, effects_sem = fit_panel_model(
    SEMPanelFE, formula, panel, W, model=3
)
display(summary_sem)
display(effects_sem)
display(diagnostics_table(idata_sem, ["lam", "beta", "sigma"]))
show_trace(idata_sem, ["lam", "sigma"], "SEMPanelFE trace")

## Fit: SDMPanelFE

In [ ]:
sdm_panel, idata_sdm, summary_sdm, effects_sdm = fit_panel_model(
    SDMPanelFE, formula, panel, W, model=3
)
display(summary_sdm)
display(effects_sdm)
display(diagnostics_table(idata_sdm, ["rho", "beta", "sigma"]))
show_trace(idata_sdm, ["rho", "sigma"], "SDMPanelFE trace")

## Fit: SDEMPanelFE

In [ ]:
sdem_panel, idata_sdem, summary_sdem, effects_sdem = fit_panel_model(
    SDEMPanelFE, formula, panel, W, model=3
)
display(summary_sdem)
display(effects_sdem)
display(diagnostics_table(idata_sdem, ["lam", "beta", "sigma"]))
show_trace(idata_sdem, ["lam", "sigma"], "SDEMPanelFE trace")

## Panel FE Spatial Specification Diagnostics (New API)

For fixed-effects panel models, the new diagnostics API provides targeted LM/LR tests for omitted spatial structure.

The examples below use:
- `OLSPanelFE.spatial_specification_tests()`
- `SARPanelFE.spatial_specification_tests()`
- `SEMPanelFE.spatial_specification_tests()`

Each entry is returned as a `DiagnosticResult` with consistent fields (`name`, `statistic`, `pvalue`, `extra`).

In [ ]:
def panel_diagnostic_to_row(result):
    row = {
        "diagnostic": result.name,
        "statistic": float(result.statistic),
        "pvalue": float(result.pvalue),
    }
    if result.extra:
        for k, v in result.extra.items():
            row[k] = float(v) if isinstance(v, (int, float, np.number)) else v
    return row


panel_spec_models = {
    "OLSPanelFE": ols_panel,
    "SARPanelFE": sar_panel,
    "SEMPanelFE": sem_panel,
}

panel_spec_tables = {}
for model_name, model_obj in panel_spec_models.items():
    tests = model_obj.spatial_specification_tests()
    table = pd.DataFrame([panel_diagnostic_to_row(dr) for dr in tests.values()])
    panel_spec_tables[model_name] = table.sort_values("diagnostic").reset_index(drop=True)

for model_name, table in panel_spec_tables.items():
    print(model_name)
    display(table)
    print()

## Random-Effects Panel Models

The random-effects models replace deterministic unit fixed effects with latent unit intercepts $\alpha_i \sim N(0, \sigma_\alpha^2)$. That lets the notebook estimate the variance of unit heterogeneity directly and compare FE and RE behavior on the same synthetic balanced panel.

### Fit: OLSPanelRE

In [ ]:
ols_panel_re, idata_ols_re, summary_ols_re, effects_ols_re = fit_panel_model(
    OLSPanelRE, formula, panel, W, model=0
)
display(summary_ols_re)
display(effects_ols_re)
display(diagnostics_table(idata_ols_re, ["beta", "sigma", "sigma_alpha"]))
show_trace(idata_ols_re, ["sigma", "sigma_alpha"], "OLSPanelRE trace")

### Fit: SARPanelRE

In [ ]:
sar_panel_re, idata_sar_re, summary_sar_re, effects_sar_re = fit_panel_model(
    SARPanelRE, formula, panel, W, model=0
)
display(summary_sar_re)
display(effects_sar_re)
display(diagnostics_table(idata_sar_re, ["rho", "beta", "sigma", "sigma_alpha"]))
show_trace(idata_sar_re, ["rho", "sigma", "sigma_alpha"], "SARPanelRE trace")

### Fit: SEMPanelRE

In [ ]:
sem_panel_re, idata_sem_re, summary_sem_re, effects_sem_re = fit_panel_model(
    SEMPanelRE, formula, panel, W, model=0
)
display(summary_sem_re)
display(effects_sem_re)
display(diagnostics_table(idata_sem_re, ["lam", "beta", "sigma", "sigma_alpha"]))
show_trace(idata_sem_re, ["lam", "sigma", "sigma_alpha"], "SEMPanelRE trace")

## MATLAB-Style Diagnostics Across FE and RE Models

In [ ]:
panel_models = {
    "OLSPanelFE": ols_panel,
    "SARPanelFE": sar_panel,
    "SEMPanelFE": sem_panel,
    "SDMPanelFE": sdm_panel,
    "SDEMPanelFE": sdem_panel,
    "OLSPanelRE": ols_panel_re,
    "SARPanelRE": sar_panel_re,
    "SEMPanelRE": sem_panel_re,
}


def _last_value(x):
    arr = np.asarray(x)
    if arr.size == 0:
        return np.nan
    return float(arr.reshape(-1)[-1])


rows = []
for name, m in panel_models.items():
    het = m.heteroskedasticity_diagnostics(arch_lags=[1, 2, 4])
    ac = m.autocorrelation_diagnostics(lags=[1, 2, 4])
    out = m.outlier_diagnostics()
    pdiag = m.panel_diagnostics()

    bp = het["bpagan"]
    arch = het["arch"]
    cd = pdiag["pesaran_cd"]

    rows.append({
        "model": name,
        "bpagan_p": float(bp.pvalue),
        "arch_lag4_p": _last_value(arch.pvalue),
        "lb_lag4_p": _last_value(ac.pvalue),
        "pesaran_cd": float(cd.statistic),
        "pesaran_cd_p": float(cd.pvalue),
        "n_leverage_flags": int(len(out["leverage_idx"])),
        "n_rstudent_flags": int(len(out["rstudent_idx"])),
        "n_dffit_flags": int(len(out["dffit_idx"])),
        "n_dfbeta_flags": int(out["dfbeta_idx"].shape[0]),
    })

diag_panel_table = pd.DataFrame(rows).sort_values("model").reset_index(drop=True)
display(diag_panel_table)

## Model Comparison (WAIC and LOO) Across FE and RE Models

In [ ]:
idata_dict = {
    "OLSPanelFE": idata_ols,
    "SARPanelFE": idata_sar,
    "SEMPanelFE": idata_sem,
    "SDMPanelFE": idata_sdm,
    "SDEMPanelFE": idata_sdem,
    "OLSPanelRE": idata_ols_re,
    "SARPanelRE": idata_sar_re,
    "SEMPanelRE": idata_sem_re,
}


def _has_loglik(idata):
    if "log_likelihood" not in idata.groups():
        return False
    return len(list(idata.log_likelihood.data_vars)) > 0


valid_for_compare = {name: idata for name, idata in idata_dict.items() if _has_loglik(idata)}
missing_loglik = [name for name, idata in idata_dict.items() if not _has_loglik(idata)]

if missing_loglik:
    print("Skipped (no log_likelihood group):", ", ".join(missing_loglik))

if len(valid_for_compare) < 2:
    print("Need at least two models with valid log_likelihood for WAIC/LOO compare.")
else:
    for ic in ("waic", "loo"):
        try:
            cmp = az.compare(valid_for_compare, ic=ic, method="BB-pseudo-BMA")
            print(f"{ic.upper()} comparison")
            display(cmp)
        except Exception as e:
            print(f"{ic.upper()} comparison not available: {type(e).__name__}: {e}")

# Always provide per-model IC attempts so the cell remains informative.
rows = []
for name, idata in idata_dict.items():
    row = {"model": name}

    try:
        waic_res = az.waic(idata)
        row["elpd_waic"] = float(waic_res.elpd_waic)
        row["p_waic"] = float(waic_res.p_waic)
    except Exception as e:
        row["elpd_waic"] = np.nan
        row["p_waic"] = np.nan
        row["waic_error"] = str(e).split("\n", 1)[0]

    try:
        loo_res = az.loo(idata)
        row["elpd_loo"] = float(loo_res.elpd_loo)
        row["p_loo"] = float(loo_res.p_loo)
    except Exception as e:
        row["elpd_loo"] = np.nan
        row["p_loo"] = np.nan
        row["loo_error"] = str(e).split("\n", 1)[0]

    rows.append(row)

ic_table = pd.DataFrame(rows).sort_values("model").reset_index(drop=True)
display(ic_table)

: 

## Notes

- This notebook uses modest draw counts for speed. For real analyses, increase `draws` and `tune`.
- If you see divergences or tree-depth warnings, increase `target_accept` (for example 0.95-0.99).
- For production inference, validate sensitivity to FE specification (`model=0/1/2/3`) and priors.